In [1]:
%pip install -qU langchain langchain-openai langchain-community langchain-text-splitters pypdf python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()

# OpenAI API 키가 설정되었는지 확인
if os.getenv("OPENAI_API_KEY"):
    print("✅ OpenAI API 키가 설정되었습니다.")
else:
    print("❌ OpenAI API 키가 설정되지 않았습니다. .env 파일을 확인해주세요.")

✅ OpenAI API 키가 설정되었습니다.


In [6]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 경로 설정 (OpenAI RAG 실습과 동일한 문서 사용)
file_path = "../../docs/DeepSeek_OCR_paper.pdf"

# PDF 로더 초기화 및 문서 로드
loader = PyPDFLoader(file_path)
docs = loader.load()

print(f"📄 로드된 문서 수: {len(docs)} 페이지")

📄 로드된 문서 수: 22 페이지


In [7]:
# 첫 번째 페이지 내용 확인
print(f"📝 첫 번째 페이지 내용 (처음 500자):\n")
print(docs[0].page_content[:500])
print("\n" + "="*50)
print(f"\n📋 메타데이터: {docs[0].metadata}")

📝 첫 번째 페이지 내용 (처음 500자):

DeepSeek-OCR: Contexts Optical Compression
Haoran Wei, Yaofeng Sun, Yukun Li
DeepSeek-AI
Abstract
We present DeepSeek-OCR as an initial investigation into the feasibility of compressing long
contexts via optical 2D mapping. DeepSeek-OCR consists of two components: DeepEncoder
and DeepSeek3B-MoE-A570M as the decoder. Specifically, DeepEncoder serves as the core
engine, designed to maintain low activations under high-resolution input while achieving high
compression ratios to ensure an optimal and


📋 메타데이터: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-10-17T08:28:51+00:00', 'author': '', 'keywords': '', 'moddate': '2025-10-17T08:28:51+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../../docs/DeepSeek_OCR_paper.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1'}


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 설정
# - chunk_size: 각 청크의 최대 문자 수
# - chunk_overlap: 청크 간 겹치는 문자 수 (문맥 유지를 위해)
# - add_start_index: 원본 문서에서의 시작 위치 추적

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # 청크 크기 (문자 수)
    chunk_overlap=200,    # 청크 간 겹침 (문자 수)
    add_start_index=True  # 원본 문서에서의 인덱스 추적
)

# 문서 분할
all_splits = text_splitter.split_documents(docs)

print(f"✂️ 원본 문서 {len(docs)}페이지가 {len(all_splits)}개의 청크로 분할되었습니다.")

✂️ 원본 문서 22페이지가 73개의 청크로 분할되었습니다.


In [9]:
# 분할된 청크 샘플 확인
print("📦 첫 번째 청크:")
print(f"내용: {all_splits[0].page_content[:300]}...")
print(f"\n메타데이터: {all_splits[0].metadata}")
print(f"\n청크 길이: {len(all_splits[0].page_content)} 문자")

📦 첫 번째 청크:
내용: DeepSeek-OCR: Contexts Optical Compression
Haoran Wei, Yaofeng Sun, Yukun Li
DeepSeek-AI
Abstract
We present DeepSeek-OCR as an initial investigation into the feasibility of compressing long
contexts via optical 2D mapping. DeepSeek-OCR consists of two components: DeepEncoder
and DeepSeek3B-MoE-A570...

메타데이터: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-10-17T08:28:51+00:00', 'author': '', 'keywords': '', 'moddate': '2025-10-17T08:28:51+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../../docs/DeepSeek_OCR_paper.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'start_index': 0}

청크 길이: 918 문자


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 인메모리 벡터 저장소 생성 및 문서 인덱싱
vector_store = InMemoryVectorStore.from_documents(
    documents=all_splits,
    embedding=embeddings
)

print("✅ 벡터 저장소 생성 완료!")